# ACF for regularly-sampled signals: Pastas vs acf_nufft

This notebook demonstrates the classic-FFT fast path of `acf_nufft` for
**regularly-sampled** series: `compute_acf_regular_fft`,
`compute_acf_rectangle_fft`, and `compute_acf_gaussian_fft` (see
`src/acf_nufft/fft_acf.py`), on five synthetic signal types:
sine, noisy sine, exponential decay, constant (degenerate case), and square wave.

For the **irregularly-sampled** case (NUFFT path) see
`notebook/pastas_vs_nufftact.ipynb`.

In [ ]:
!pip install -q "acf-nufft[benchmark] @ git+https://github.com/jecampagne/nufftacf.git"


In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import pastas as ps

from acf_nufft import (
    compute_acf_regular_fft,
    compute_acf_rectangle_fft,
    compute_acf_gaussian_fft,
)

import acf_nufft
print(f'acf_nufft : {acf_nufft.__version__}')
print(f'pastas    : {ps.__version__}')


In [6]:
import matplotlib as mpl
mpl.rcParams.update({
    'font.size'         : 13,
    'axes.titlesize'    : 14,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 14,
    'xtick.labelsize'   : 14,
    'ytick.labelsize'   : 14,
    'legend.fontsize'   : 12,
    'figure.titlesize'  : 16,
    'figure.titleweight': 'bold',
})

## Test series

Five synthetic signal types on a regular daily grid.

In [ ]:
def generate_series(n_days: int, signal_type: str = "sinus", **params) -> pd.Series:
    """Generate a regularly-sampled (delta_t=1 day) series.

    signal_type: one of "sinus", "sinus_bruit", "exponentielle", "constant", "carre".
    """
    t = np.arange(n_days, dtype=float)
    if signal_type == "sinus":
        f = params.get("f", 1.0 / (n_days / 10))
        data = np.sin(2 * np.pi * f * t)
    elif signal_type == "sinus_bruit":
        f = params.get("f", 1.0 / (n_days / 10))
        rng = np.random.default_rng(params.get("seed", 0))
        data = np.sin(2 * np.pi * f * t) + rng.standard_normal(n_days) * 0.1
    elif signal_type == "exponentielle":
        tau_0 = params.get("tau_0", n_days / 5)
        data = np.exp(-t / tau_0)
    elif signal_type == "constant":
        # Degenerate case: zero variance -> ACF is mathematically undefined.
        # Verifies that the package returns NaN everywhere rather than crashing.
        data = np.ones(n_days)
    elif signal_type == "carre":
        f = params.get("f", 1.0 / (n_days / 10))
        data = np.sign(np.sin(2 * np.pi * f * t))
    else:
        raise ValueError(f"Unknown signal_type {signal_type!r}")
    idx = pd.date_range("2000-01-01", periods=n_days, freq="D")
    return pd.Series(data=data, index=idx, name=f"STS_{signal_type}")


SIGNAL_TYPES = ["sinus", "sinus_bruit", "exponentielle", "constant", "carre"]
N_days = 3650


## Pastas vs acf_nufft comparison

For each signal type, the ACF is computed with Pastas and with `acf_nufft` (FFT path)
for all three bin methods (`regular`, `rectangle`, `gaussian`), then overlaid.

In [ ]:
FFT_FUNCS = {
    "regular"  : compute_acf_regular_fft,
    "rectangle": compute_acf_rectangle_fft,
    "gaussian" : compute_acf_gaussian_fft,
}


def compute_comparison(sts: pd.Series, lags_max: int = 365, bin_width: float = 0.5):
    """Compute Pastas and acf_nufft (fft) for all three bin methods on one series."""
    lags = np.arange(0.0, lags_max + 1.0)
    t = (sts.index - sts.index[0]) / pd.Timedelta(days=1)
    t = t.to_numpy(dtype=float)
    x = sts.to_numpy()

    out = {}
    for method in ["regular", "rectangle", "gaussian"]:
        t0 = time.time()
        acf_pastas = ps.stats.acf(
            sts, lags=lags, bin_method=method, bin_width=bin_width, min_obs=0
        )
        dt_pastas = time.time() - t0

        func = FFT_FUNCS[method]
        t0 = time.time()
        if method == "regular":
            c, b = func(lags, t, x)
        else:
            c, b = func(lags, t, x, bin_width=bin_width)
        dt_fft = time.time() - t0

        out[method] = dict(
            pastas_lags=acf_pastas.index.days.to_numpy(),
            pastas_c=acf_pastas.to_numpy(),
            fft_lags=lags,
            fft_c=c,
            t_pastas=dt_pastas,
            t_fft=dt_fft,
        )
    return out


def plot_comparison(out: dict, title: str):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    for ax, method in zip(axes, ["regular", "rectangle", "gaussian"]):
        r = out[method]
        ax.plot(
            r["pastas_lags"], r["pastas_c"],
            "-", color="tab:blue", lw=2,
            label=f"Pastas ({method}) -- {r['t_pastas']*1000:.1f} ms",
        )
        ax.plot(
            r["fft_lags"], r["fft_c"],
            "--", color="tab:red", lw=2,
            label=f"acf_nufft fft ({method}) -- {r['t_fft']*1000:.2f} ms",
        )
        ax.set_xlabel("Lag [days]")
        ax.set_title(method)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
for signal_type in SIGNAL_TYPES:
    sts = generate_series(N_days, signal_type)
    print(f"--- {signal_type} (N={N_days}) ---")
    if signal_type == "constant":
        # Zero-variance signal: verify the package returns NaN everywhere.
        t = np.arange(N_days, dtype=float)
        x = sts.to_numpy()
        lags = np.arange(0.0, 366.0)
        for method, func in FFT_FUNCS.items():
            kwargs = {} if method == "regular" else dict(bin_width=0.5)
            with np.errstate(divide="ignore", invalid="ignore"):
                c, b = func(lags, t, x, **kwargs)
            assert np.all(np.isnan(c)), f"{method}: expected all-NaN for constant signal"
        print("  OK: ACF is all-NaN for the constant (zero-variance) signal.")
        continue
    out = compute_comparison(sts)
    plot_comparison(out, f"{signal_type}  (N={N_days})")
